<a href="https://colab.research.google.com/github/Sanjjjayyy/haystack-cookbook/blob/main/notebooks/youtube_transcript_rag_haystack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouTube Transcript RAG with Haystack and HuggingFace

> This cookbook shows how to build a RAG pipeline over any YouTube video
> transcript using [Haystack](https://github.com/deepset-ai/haystack).

## Overview

## What this cookbook does

- [Fetches the transcript of any public YouTube video automatically](#step-1-fetch-youtube-transcript)
- [Splits it into searchable chunks](#step-2-split-and-index-transcript-using-haystack)
- [Embeds chunks using BAAI/bge-base-en-v1.5](#step-2-split-and-index-transcript-using-haystack)
- [Retrieves the most relevant chunks using semantic search](#step-3-build-rag-query-pipeline-using-haystack)
- [Generates accurate answers using Qwen2.5 via HuggingFace Inference API (free)](#step-3-build-rag-query-pipeline-using-haystack)

## Components used
- [YouTubeTranscriptApi](https://github.com/jdepoix/youtube-transcript-api): to fetch video transcripts
- [DocumentSplitter](https://docs.haystack.deepset.ai/docs/documentsplitter): to split transcript into overlapping chunks
- [SentenceTransformersDocumentEmbedder](https://docs.haystack.deepset.ai/docs/sentencetransformersdocumentembedder): to embed chunks using `BAAI/bge-base-en-v1.5`
- [InMemoryDocumentStore](https://docs.haystack.deepset.ai/docs/inmemorydocumentstore): to store and search embedded chunks
- [InMemoryEmbeddingRetriever](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever): to retrieve relevant chunks for a query
- [InferenceClient](https://huggingface.co/docs/huggingface_hub/guides/inference): to generate answers using `Qwen2.5-7B-Instruct`



## Install the dependencies

In [ ]:
!pip install haystack-ai youtube-transcript-api sentence-transformers -q

In [ ]:
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceAPIChatGenerator
from haystack.components.preprocessors import DocumentSplitter
from haystack.dataclasses import ChatMessage
from huggingface_hub import InferenceClient
from youtube_transcript_api import YouTubeTranscriptApi
import os

## Setting up your HuggingFace API Token

This cookbook uses HuggingFace's free Inference API to generate answers.
You'll need a free HuggingFace account and API token to proceed.

### Step 1 — Create a free HuggingFace account
Go to [huggingface.co/join](https://huggingface.co/join) and sign up for free.

### Step 2 — Generate an API token
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Click **"New token"**
3. Give it any name (e.g. `haystack-notebook`)
4. Set the role to **"Read"**
5. Click **"Generate a token"** and copy it

### Step 3 — Add token to Colab Secrets
1. Click the ** key icon** in the left sidebar of Colab
2. Click **"Add new secret"**
3. Set the name to exactly `HF_TOKEN`
4. Paste your token as the value
5. Toggle the **notebook access** switch to ON


In [ ]:
from google.colab import userdata
import os

os.environ["HF_API_TOKEN"] = userdata.get('HF_TOKEN')

## Step 1 — Fetch YouTube Transcript



We use `youtube-transcript-api` to fetch the full transcript of any public
YouTube video. The transcript is joined into a single string and wrapped
into a Haystack [`Document`](https://docs.haystack.deepset.ai/docs/data-classes#document)
object — the core data structure Haystack uses to represent text content
throughout the entire pipeline.

> A `Document` in Haystack can hold text content, metadata, and embeddings
> all in one place — making it easy to pass data between pipeline components.

In [ ]:
def get_transcript(video_url):

    # Extract video ID from URL
    video_id=video_url.split("v=")[-1].split("&")[0]

    # Fetch transcript using new API
    ytt_api=YouTubeTranscriptApi()
    transcript=ytt_api.fetch(video_id)

    # Join all text chunks into one document
    full_text=" ".join([entry.text for entry in transcript])

    return full_text

# Enter any YouTube URL here
video_url=input("Enter YouTube URL: ")
transcript_text=get_transcript(video_url)

print(f"Transcript fetched successfully!")
print(f"Total characters: {len(transcript_text)}")
print(f"\nPreview:\n{transcript_text[:300]}...")

## Step 2 — Split and Index Transcript using Haystack

This step uses three Haystack components to prepare the transcript for retrieval.

### DocumentSplitter
[`DocumentSplitter`](https://docs.haystack.deepset.ai/docs/documentsplitter)
breaks the transcript `Document` into smaller chunks of **150 words** with
**20 word overlap**.

Haystack's `DocumentSplitter` supports splitting by `word`, `sentence`, or
`passage` — we use `word` here for consistent chunk sizes. The overlap ensures
context is preserved at chunk boundaries.

### SentenceTransformersDocumentEmbedder
[`SentenceTransformersDocumentEmbedder`](https://docs.haystack.deepset.ai/docs/sentencetransformersdocumentembedder)
converts each chunk into a dense vector embedding using `BAAI/bge-base-en-v1.5`
— a top ranked model on the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).

Haystack's embedder automatically adds the embedding to each `Document` object's
`embedding` field, keeping text and its vector representation together.

### InMemoryDocumentStore
[`InMemoryDocumentStore`](https://docs.haystack.deepset.ai/docs/inmemorydocumentstore)
stores all embedded `Document` objects in memory for fast similarity search.

`InMemoryDocumentStore` is ideal for lightweight single-video demos like this one.

For larger scale use cases with multiple videos haystack supports many document stores for production use cases — including
[Qdrant](https://haystack.deepset.ai/integrations/qdrant),
[Weaviate](https://haystack.deepset.ai/integrations/weaviate), and
[Elasticsearch](https://haystack.deepset.ai/integrations/elasticsearch).



In [ ]:
# Initialize document store
document_store=InMemoryDocumentStore()

# Convert to Haystack Document
docs=[Document(content=transcript_text)]

# Split into chunks
splitter=DocumentSplitter(
    split_by="word",
    split_length=150,
    split_overlap=20
)

split_docs=splitter.run(documents=docs)["documents"]

# Embed chunks
embedder=SentenceTransformersDocumentEmbedder(
    model="BAAI/bge-base-en-v1.5"
)
embedder.warm_up()
embedded_docs=embedder.run(documents=split_docs)["documents"]

# Write to store
document_store.write_documents(embedded_docs)

# Verify documents are stored correctly
print(f" {document_store.count_documents()} chunks indexed successfully!")

## Step 3 — Build RAG Query Pipeline

The query pipeline runs three steps in sequence:

### 1. Embed the question
The question is embedded using
[`SentenceTransformersTextEmbedder`](https://docs.haystack.deepset.ai/docs/sentencetransformerstextembedder)
with the same `BAAI/bge-base-en-v1.5` model used during indexing.

> It is critical to use the **same model** for both document and query
> embedding — different models produce vectors in different spaces, making
> similarity search meaningless.

### 2. Retrieve relevant chunks
[`InMemoryEmbeddingRetriever`](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever)
finds the **top 5 most similar chunks** using cosine similarity between the
question embedding and all stored chunk embeddings.

This is semantic search — it understands *meaning*, not just keyword matches.

### 3. Generate the answer
The 5 retrieved chunks are passed as context to `Qwen2.5-7B-Instruct` via
HuggingFace's free
[Inference API](https://huggingface.co/docs/api-inference/index).

The model is explicitly instructed to answer **only** based on the provided
transcript context — this prevents hallucinations and keeps answers grounded
in the video content.

In [ ]:
# Initialize components
text_embedder=SentenceTransformersTextEmbedder(
    model="BAAI/bge-base-en-v1.5"
)
text_embedder.warm_up()

retriever=InMemoryEmbeddingRetriever(
    document_store=document_store,
    top_k=5
)

client=InferenceClient(
    model="Qwen/Qwen2.5-7B-Instruct",
    token=os.environ["HF_API_TOKEN"]
)

def ask(question):

    # Embed question
    embedded_question=text_embedder.run(text=question)

    # Retrieve relevant chunks
    retrieved=retriever.run(
        query_embedding=embedded_question["embedding"]
    )

    # Build context
    context="\n".join([doc.content for doc in retrieved["documents"]])

    # Build prompt
    prompt = f"""You are a helpful assistant that answers questions
             based on YouTube video transcripts.

Context:{context}

Question:{question}

Answer based only on the transcript content above.
"""
    # Generate answer
    result=client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )

    print(f"Question: {question}")
    print(f"\nAnswer: {result.choices[0].message.content}")
    print("\n" + "─"*60 + "\n")

print("Pipeline ready!")

## Step 4 — Ask Questions

The pipeline is ready! Pass any YouTube URL and ask questions about the video.

> **Tips for best results:**
> - Ask specific questions rather than vague ones
> - Conceptual questions like *"How does X work?"* perform better than
>   opinion-based ones
> - The answer quality depends on how clearly the topic is covered in the video

Feel free to swap in any YouTube URL and ask your own questions below!

In [ ]:
# Try your own YouTube URL and questions!
ask("What is the main topic of this video?")
ask("What are the key concepts explained?")
ask("What prerequisites does the speaker recommend?")

## You built a YouTube RAG Pipeline!

In this cookbook you built a complete RAG pipeline that can answer questions about any YouTube video using Haystack and HuggingFace — completely free.

### Further Reading
- [Haystack Documentation](https://docs.haystack.deepset.ai)
- [Haystack Tutorials](https://haystack.deepset.ai/tutorials)
- [HuggingFace Inference API](https://huggingface.co/docs/api-inference/index)
- [Other Haystack Cookbooks](https://github.com/deepset-ai/haystack-cookbook)